In [1]:
import folium
import math

# --- 1. FUNÇÃO DE CÁLCULO (A mesma anterior) ---
def geo_project_point(A, C, B):
    # Conversão aproximada lat/lon para metros em Cametá
    lat_to_m = 111320 
    lat_media_rad = math.radians(A[0])
    lon_to_m = 111320 * math.cos(lat_media_rad)

    # Converter para XY (Metros)
    def to_xy(ponto):
        return ((ponto[1] - A[1]) * lon_to_m, (ponto[0] - A[0]) * lat_to_m)

    A_m, C_m, B_m = (0,0), to_xy(C), to_xy(B)

    # Cálculo Vetorial
    vec_AC = (C_m[0] - A_m[0], C_m[1] - A_m[1])
    vec_AB = (B_m[0] - A_m[0], B_m[1] - A_m[1])
    
    dot = (vec_AB[0] * vec_AC[0]) + (vec_AB[1] * vec_AC[1])
    mag_sq = (vec_AC[0]**2) + (vec_AC[1]**2)
    t = dot / mag_sq

    # Ponto P em metros
    Px_m = A_m[0] + t * vec_AC[0]
    Py_m = A_m[1] + t * vec_AC[1]

    # Converter de volta para Lat/Lon
    P_lat = A[0] + (Py_m / lat_to_m)
    P_lon = A[1] + (Px_m / lon_to_m)
    
    return (P_lat, P_lon)

# --- 2. DADOS DE ENTRADA ---
A = (-2.246183, -49.506388) # Início Base
C = (-2.242185, -49.498365) # Fim Base
B = (-2.250035, -49.502177) # Ponto Topo

# Calcula o ponto P
P = geo_project_point(A, C, B)

# --- 3. GERAÇÃO DO MAPA (FOLIUM) ---

# Cria o mapa centrado na média dos pontos
center_lat = (A[0] + B[0] + C[0]) / 3
center_lon = (A[1] + B[1] + C[1]) / 3
m = folium.Map(location=[center_lat, center_lon], zoom_start=15, tiles="OpenStreetMap")

# Dicionário para facilitar plotagem
pontos = {
    "A (Início)": {"coord": A, "color": "blue", "icon": "play"},
    "C (Fim)":    {"coord": C, "color": "blue", "icon": "stop"},
    "B (Topo)":   {"coord": B, "color": "red",  "icon": "info-sign"},
    "P (Projeção)": {"coord": P, "color": "green", "icon": "screenshot"}
}

# Adicionar Marcadores
for nome, dados in pontos.items():
    folium.Marker(
        location=dados["coord"],
        popup=f"<b>{nome}</b><br>{dados['coord']}",
        icon=folium.Icon(color=dados["color"], icon=dados["icon"])
    ).add_to(m)

# Adicionar Linhas (Polylines)
# 1. Aresta Base (A -> C) em Azul
folium.PolyLine([A, C], color="blue", weight=4, opacity=0.7, tooltip="Aresta Base").add_to(m)

# 2. Lados do Triângulo (A -> B e C -> B) pontilhados para contexto
folium.PolyLine([A, B], color="gray", weight=2, dash_array='5, 10').add_to(m)
folium.PolyLine([C, B], color="gray", weight=2, dash_array='5, 10').add_to(m)

# 3. A LINHA ORTOGONAL (B -> P) em Vermelho
folium.PolyLine([B, P], color="red", weight=4, opacity=0.9, tooltip="Ortogonal (Altura)").add_to(m)

# --- 4. SALVAR ARQUIVO ---
arquivo_saida = "triangulo_cameta.html"
m.save(arquivo_saida)

print(f"Mapa gerado com sucesso!")
print(f"Ponto P calculado: {P}")
print(f"Abra o arquivo '{arquivo_saida}' no seu navegador para ver o mapa.")

Mapa gerado com sucesso!
Ponto P calculado: (-2.2452697210615837, -49.50455527440648)
Abra o arquivo 'triangulo_cameta.html' no seu navegador para ver o mapa.
